<a href="https://colab.research.google.com/github/eva-beresin/ML-for-AML/blob/main/BMED_3201_AML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

found_shared_path = '/content/drive/.shortcut-targets-by-id/12xPmaWP-eerNy9aA7G8cUXyiwEOHKWZ_/BMED3201_Projects/'
aml_csv_dir = os.path.join(found_shared_path, 'AML_Classification', 'CSV')

print(f"Target directory: {aml_csv_dir}")

aml_csv_files = []
if os.path.exists(aml_csv_dir):
    for file in sorted(os.listdir(aml_csv_dir)):
        if file.lower().endswith('.csv'):
            aml_csv_files.append(os.path.join(aml_csv_dir, file))

    print(f"Found {len(aml_csv_files)} AML Classification CSV files.")
    if aml_csv_files:
        print("\nPreview of the first file:")
        df_aml_preview = pd.read_csv(aml_csv_files[0])
        display(df_aml_preview.head())
else:
    print("Error: Could not locate the 'AML_Classification/CSV' directory.")

Target directory: /content/drive/.shortcut-targets-by-id/12xPmaWP-eerNy9aA7G8cUXyiwEOHKWZ_/BMED3201_Projects/AML_Classification/CSV
Found 2872 AML Classification CSV files.

Preview of the first file:


,FS Lin,SS Log,FL1 Log,FL2 Log,FL3 Log,FL4 Log,FL5 Log
0,427,0.452041,0.140279,0.213405,0.683650,0.160077,0.140279
1,917,0.620252,0.253195,0.276627,0.609630,0.203733,0.257787
2,890,0.620252,0.248659,0.184568,0.592770,0.140279,0.140279
3,855,0.656414,0.246862,0.248659,0.530069,0.167291,0.144721
4,921,0.629973,0.212638,0.265240,0.598100,0.212638,0.162236


In [ ]:
aml_preview_stats = df_aml_preview.describe()
aml_preview_stats

,FS Lin,SS Log,FL1 Log,FL2 Log,FL3 Log,FL4 Log,FL5 Log
count,26966.000000,26966.000000,26966.000000,26966.000000,26966.000000,26966.000000,26966.000000
mean,663.982311,0.554379,0.205182,0.202277,0.589269,0.181832,0.161980
std,218.849759,0.095017,0.052198,0.048688,0.107338,0.043249,0.026069
min,150.000000,0.219672,0.140279,0.140279,0.140279,0.140279,0.140279
25%,464.000000,0.451106,0.144425,0.157195,0.546297,0.140279,0.140279
50%,713.000000,0.602538,0.211116,0.201595,0.606085,0.172787,0.153350
75%,848.000000,0.629090,0.240654,0.237166,0.663450,0.208861,0.177072
max,1022.000000,0.767618,0.586544,0.576743,0.859109,0.501908,0.435142


In [ ]:
output_stats_path = os.path.join(found_shared_path, 'AML_Classification', 'aml_preview_descriptive_stats.csv')
aml_preview_stats.to_csv(output_stats_path)
print(f"Saved stats to: {output_stats_path}")

Successfully exported descriptive statistics to: /content/drive/.shortcut-targets-by-id/12xPmaWP-eerNy9aA7G8cUXyiwEOHKWZ_/BMED3201_Projects/AML_Classification/aml_preview_descriptive_stats.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the plotting style
sns.set_theme(style="whitegrid")

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of FS Lin
sns.histplot(df_aml_preview['FS Lin'], bins=100, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of FS Lin')
axes[0].set_xlabel('FS Lin')
axes[0].set_ylabel('Count')

# 2. Distribution of SS Log
sns.histplot(df_aml_preview['SS Log'], bins=100, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of SS Log')
axes[1].set_xlabel('SS Log')
axes[1].set_ylabel('Count')

# 3. FS Lin vs SS Log (Scatter / Density Plot)
# Using scatterplot with low alpha (transparency) or hexbin due to many points
axes[2].hexbin(df_aml_preview['FS Lin'], df_aml_preview['SS Log'], gridsize=100, cmap='Blues', mincnt=1)
axes[2].set_title('FS Lin vs SS Log Density')
axes[2].set_xlabel('FS Lin')
axes[2].set_ylabel('SS Log')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. FS Lin Distribution
sns.histplot(df_aml_preview['FS Lin'], bins=100, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of FS Lin')
axes[0].set_xlabel('FS Lin')
axes[0].set_ylabel('Count')

# 2. SS Log Distribution
sns.histplot(df_aml_preview['SS Log'], bins=100, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of SS Log')
axes[1].set_xlabel('SS Log')
axes[1].set_ylabel('Count')

# 3. FS vs SS Joint Scatter/Density Plot
axes[2].hexbin(df_aml_preview['FS Lin'], df_aml_preview['SS Log'], gridsize=100, cmap='Purples', mincnt=1)
axes[2].set_title('FS Lin vs SS Log Joint Density')
axes[2].set_xlabel('FS Lin')
axes[2].set_ylabel('SS Log')

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# Prepare lists to accumulate computed statistical summaries
consolidated_data = []

print(f"Extracting patient-level statistics from {len(aml_csv_files)} AML CSV files...")

# Iterate through each patient file
for filepath in tqdm(aml_csv_files):
    try:
        # Read the cell observations
        cell_data = pd.read_csv(filepath)
        sample_id = os.path.basename(filepath).replace('.CSV', '').replace('.csv', '')

        # Initialize our feature row with the unique identifier
        sample_features = {'sample_id': sample_id}

        # Compute statistical descriptors for each channel
        for channel in cell_data.columns:
            if cell_data[channel].dtype in [np.float64, np.int64]:
                values = cell_data[channel].dropna()
                if len(values) > 0:
                    sample_features[f'{channel}_mean'] = np.mean(values)
                    sample_features[f'{channel}_std'] = np.std(values)
                    sample_features[f'{channel}_median'] = np.median(values)
                    sample_features[f'{channel}_min'] = np.min(values)
                    sample_features[f'{channel}_max'] = np.max(values)
                    sample_features[f'{channel}_25th'] = np.percentile(values, 25)
                    sample_features[f'{channel}_75th'] = np.percentile(values, 75)

        consolidated_data.append(sample_features)
    except Exception as e:
        # Skip any unreadable or corrupted files
        continue

# Build the final consolidated DataFrame
df_features = pd.DataFrame(consolidated_data)
print(f"\nConsolidation complete!")
print(f"Feature matrix dimensions: {df_features.shape} (Patients, Features)")
display(df_features.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Split df_features into training (80%) and testing (20%) datasets
# We'll use a random_state to ensure the split is reproducible
train_df, test_df = train_test_split(df_features, test_size=0.2, random_state=42)

print(f"Training set shape: {train_df.shape} (Patients, Features)")
print(f"Testing set shape:  {test_df.shape} (Patients, Features)")

# Display a preview of the training partition
display(train_df.head())

In [ ]:
import os

# Let's inspect the files in the parent directory and search for label/metadata files
project_root = '/content/drive/.shortcut-targets-by-id/12xPmaWP-eerNy9aA7G8cUXyiwEOHKWZ_/BMED3201_Projects/'
print("Searching for label files in project directory...")

for root, dirs, files in os.walk(project_root):
    for file in files:
        # Look for typical metadata filenames
        if any(keyword in file.lower() for keyword in ['label', 'metadata', 'clinical', 'patient', 'class']):
            print(f"Found potential label file: {os.path.join(root, file)}")

In [ ]:
# Cell removed after reviewing screenshot contents.

In [ ]:
import os

# Broad search for any spreadsheets, CSVs, or text files in the project workspace
print("Searching all tabular/text files in project directory for potential patient classifications...")
for root, dirs, files in os.walk(project_root):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in ['.csv', '.xlsx', '.txt']:
            print(f"File: {file} in Path: {root}")

In [ ]:
# Convert sample IDs to integers and inspect their distribution and range
# to see if the patient cohorts are cleanly partitioned.
sample_ids = sorted([int(sid) for sid in df_features['sample_id'] if sid.isdigit()])
if sample_ids:
    print(f"Total unique patient sample IDs: {len(sample_ids)}")
    print(f"Minimum ID: {min(sample_ids)}")
    print(f"Maximum ID: {max(sample_ids)}")

    # Check if there is a clean continuous range of IDs
    is_continuous = len(sample_ids) == (max(sample_ids) - min(sample_ids) + 1)
    print(f"Is the ID range fully continuous? {is_continuous}")
else:
    print("Could not parse sample IDs as integers.")

In [ ]:
# Convert sample IDs to integers and inspect their distribution and range
# to see if the patient cohorts are cleanly partitioned.
sample_ids = sorted([int(sid) for sid in df_features['sample_id'] if sid.isdigit()])
if sample_ids:
    print(f"Total unique patient sample IDs: {len(sample_ids)}")
    print(f"Minimum ID: {min(sample_ids)}")
    print(f"Maximum ID: {max(sample_ids)}")

    # Check if there is a clean continuous range of IDs
    is_continuous = len(sample_ids) == (max(sample_ids) - min(sample_ids) + 1)
    print(f"Is the ID range fully continuous? {is_continuous}")
else:
    print("Could not parse sample IDs as integers.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# Separate features and target variables using the labeled dataframes
X_train = train_df.drop(columns=['sample_id', 'label'])
y_train = train_df['label']
X_test = test_df.drop(columns=['sample_id', 'label'])
y_test = test_df['label']

print("Training classifier...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict and evaluate performance
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"\nAccuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'AML (1)']))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix visually
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal (0)', 'AML (1)'])
disp.plot(cmap='Blues', values_format='d', ax=ax)

plt.title('AML vs Normal Classification Confusion Matrix')
plt.grid(False) # Turn off grid lines over the matrix
plt.show()

In [ ]:
# Map sample IDs to binary labels based on the continuous sequence partitioning
# IDs 1 to 1436: Normal (0), IDs 1437 to 2872: AML (1)

train_df = train_df.copy()
test_df = test_df.copy()

train_df['label'] = train_df['sample_id'].astype(int).apply(lambda x: 0 if x <= 1436 else 1)
test_df['label'] = test_df['sample_id'].astype(int).apply(lambda x: 0 if x <= 1436 else 1)

print("Label distribution in Training Set:")
print(train_df['label'].value_counts())

print("\nLabel distribution in Testing Set:")
print(test_df['label'].value_counts())